# 正则化

欢迎来到本周的第二篇作业。深度学习模型具有如此大的灵活性和容量，如果训练数据集不够大，**过拟合可能是一个严重的问题**。当然，它在训练集上表现良好，但是学习的网络**不会泛化到它从未见过的新示例**！

**你将学习：** 在深度学习模型中使用正则化。

让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/deep-neural-network/supplement/QWEnZ/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1 - 依赖库](#1)
- [2 - 问题描述](#2)
- [3 - 加载数据集](#3)
- [4 - 非正则化模型](#4)
- [5 - L2 正则化](#5)
    - [练习 1 - compute_cost_with_regularization](#ex-1)
    - [练习 2 - backward_propagation_with_regularization](#ex-2)
- [6 - Dropout](#6)
    - [6.1 - 前向传播与 Dropout](#6-1)
        - [练习 3 - forward_propagation_with_dropout](#ex-3)
    - [6.2 - 反向传播与 Dropout](#6-2)
        - [练习 4 - backward_propagation_with_dropout](#ex-4)
- [7 - 结论](#7)

<a name='1'></a>
## 1 - 依赖库

In [ ]:
### v1.1

In [ ]:
# import packages
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import sklearn.datasets
import scipy.io
from reg_utils import sigmoid, relu, plot_decision_boundary, initialize_parameters, load_2D_dataset, predict_dec
from reg_utils import compute_cost, predict, forward_propagation, backward_propagation, update_parameters
from testCases import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # 设置绘图的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

<a name='2'></a>
## 2 - 问题描述

你刚刚被法国足球公司聘为人工智能专家。他们希望你推荐法国队守门员应该踢球的位置，以便法国队球员可以用头击球。 

<img src="images/field_kiank.png" style="width:600px;height:350px;">

<div style="text-align:center;color:purple;"><b>图1</b>：足球场。守门员将球踢向空中，各队球员争用头击球</div>


他们为你提供以下法国队过去 10 场比赛的 2D 数据集。

<a name='3'></a>
## 3 - 加载数据集

In [ ]:
train_X, train_Y, test_X, test_Y = load_2D_dataset()

每个点对应足球场上法国守门员从足球场左侧射门后足球运动员用头击球的位置。
- 如果圆点是蓝色的，则表示法国球员成功用头击球
- 如果圆点是红色的，则表示对方球员用头击球

**你的目标**：使用深度学习模型找到守门员在场上应该踢球的位置。

**数据集分析**：这个数据集有点嘈杂，但看起来像分隔左上半部分（蓝色）和右下半部分（红色）的对角线效果很好。 

你将首先尝试非正则化模型。然后你将学习如何对其进行正则化并决定选择哪种模型来解决法国足球公司的问题。 

<a name='4'></a>
## 4 - 非正则化模型

你将使用以下神经网络（下面已为你实现）。该模型可用于：
- -- 通过将 `lambd` 输入设置为非零值。我们使用“`lambd`”而不是“`lambda`”，因为“`lambda`”是Python中的保留关键字。 *正则化模式* 下的 
- 在*Dropout模式*中——通过将`keep_prob`设置为小于1的值

你将首先尝试没有任何正则化的模型。然后，你将实现：
- *L2 正则化* -- 函数：“`compute_cost_with_regularization()`”和“`backward_propagation_with_regularization()`”
- *Dropout* -- 函数：“`forward_propagation_with_dropout()`”和“`backward_propagation_with_dropout()`”

在每个部分中，你将使用正确的输入运行此模型，以便它调用你已实现的函数。查看下面的代码以熟悉该模型。

In [ ]:
def model(X, Y, learning_rate = 0.3, num_iterations = 30000, print_cost = True, lambd = 0, keep_prob = 1):
    """
    实现三层神经网络：LINEAR->ReLU->LINEAR->ReLU->LINEAR->sigmoid。
    
    参数：
    X -- 输入数据，形状（输入大小，示例数量）
    Y -- 真实的 "label" 向量（1 表示蓝点/0 表示红点），形状（输出大小、示例数量）
    learning_rate -- 优化的学习率
    num_iterations -- 优化循环的迭代次数
    print_cost -- 如果为 True，则打印每 10000 次迭代的代价
    lambd -- 正则化超参数，标量
    keep_prob - 在Dropout期间保持神经元活跃的概率，标量。
    
    返回：
    parameters -- 模型学习到的参数。然后可以使用它们进行预测。
    """
        
    grads = {}
    costs = []                            # 跟踪代价
    m = X.shape[1]                        # 示例数量
    layers_dims = [X.shape[0], 20, 3, 1]
    
    # 初始化参数字典。
    parameters = initialize_parameters(layers_dims)

    # 循环 (梯度下降)

    for i in range(0, num_iterations):

        # 前向传播：线性 -> ReLU -> 线性 -> ReLU -> 线性 -> sigmoid。
        if keep_prob == 1:
            a3, cache = forward_propagation(X, parameters)
        elif keep_prob < 1:
            a3, cache = forward_propagation_with_dropout(X, parameters, keep_prob)
        
        # 代价函数
        if lambd == 0:
            cost = compute_cost(a3, Y)
        else:
            cost = compute_cost_with_regularization(a3, Y, parameters, lambd)
            
        # 反向传播.
        assert (lambd == 0 or keep_prob == 1)   # 可以同时使用 L2 正则化和 Dropout， 
                                                # 但是这个作业一次只会探索一个
        if lambd == 0 and keep_prob == 1:
            grads = backward_propagation(X, Y, cache)
        elif lambd != 0:
            grads = backward_propagation_with_regularization(X, Y, cache, lambd)
        elif keep_prob < 1:
            grads = backward_propagation_with_dropout(X, Y, cache, keep_prob)
        
        # 更新参数。
        parameters = update_parameters(parameters, grads, learning_rate)
        
        # 每 10000 次迭代打印损失
        if print_cost and i % 10000 == 0:
            print("Cost after iteration {}: {}".format(i, cost))
        if print_cost and i % 1000 == 0:
            costs.append(cost)
    
    # 绘制代价图
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('iterations (x1,000)')
    plt.title("Learning rate =" + str(learning_rate))
    plt.show()
    
    return parameters

让我们在没有任何正则化的情况下训练模型，并观察训练/测试集上的准确性。

In [ ]:
parameters = model(train_X, train_Y)
print ("On the training set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

训练准确率为94.8%，测试准确率为91.5%。这是**基线模型**（你将观察正则化对此模型的影响）。运行以下代码来绘制模型的决策边界。

In [ ]:
plt.title("Model without regularization")
axes = plt.gca()
axes.set_xlim([-0.75,0.40])
axes.set_ylim([-0.75,0.65])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

未正则化的模型明显过拟合了训练集，连噪声点也被拟合。接下来介绍两种减轻过拟合的方法。

<a name='5'></a>
## 5 - L2 正则化

避免过拟合的标准方法称为 **L2 正则化**。它包括适当修改你的代价函数，来自：
$$J = -\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} \tag{1}$$
致：
$$J_{regularized} = \small \underbrace{-\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} }_\text{cross-entropy cost} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2} }_\text{L2 regularization cost} \tag{2}$$

让我们修改你的代价并观察后果。

<a name='ex-1'></a>
### 练习 1 - compute_cost_with_regularization
实现 `compute_cost_with_regularization()`，计算公式 (2) 给出的代价。要计算 $\sum\limits_k\sum\limits_j W_{k,j}^{[l]2}$ ，请使用：
```python
np.sum(np.square(Wl))
```
请注意，你必须对 $W^{[1]}$、$W^{[2]}$ 和 $W^{[3]}$ 执行此操作，然后将三项相加并乘以 $ \frac{1}{m} \frac{\lambda}{2} $。

In [ ]:
# GRADED FUNCTION: compute_cost_with_regularization

def compute_cost_with_regularization(A3, Y, parameters, lambd):
    """
    使用 L2 正则化实现代价函数。参见上面的公式（2）。
    
    参数：
    A3 -- 激活值之后，前向传播的输出，形状（输出大小，示例数量）
    Y -- "true" 标签向量，形状（输出大小，示例数量）
    parameters -- 包含模型参数的Python字典
    
    返回：
    cost - 正则化后的损失函数的值（式（2））
    """
    m = Y.shape[1]
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    W3 = parameters["W3"]
    
    cross_entropy_cost = compute_cost(A3, Y) # 这给你代价的交叉熵部分
    
    #（约1行代码）
    # L2_regularization_cost = 
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    cost = cross_entropy_cost + L2_regularization_cost
    
    return cost

In [ ]:
A3, t_Y, parameters = compute_cost_with_regularization_test_case()
cost = compute_cost_with_regularization(A3, t_Y, parameters, lambd=0.1)
print("cost = " + str(cost))

compute_cost_with_regularization_test(compute_cost_with_regularization)

当然，因为你改变了代价，所以你也必须改变反向传播！所有梯度都必须根据这个新代价进行计算。 

<a name='ex-2'></a>
### 练习 2 - backward_propagation_with_regularization
实现反向传播中所需的更改以考虑正则化。更改仅涉及 dW1、dW2 和 dW3。对于每个项，你必须添加正则化项的梯度 ($\frac{d}{dW} ( \frac{1}{2}\frac{\lambda}{m}  W^2) = \frac{\lambda}{m} W$)。

In [ ]:
# GRADED FUNCTION: backward_propagation_with_regularization

def backward_propagation_with_regularization(X, Y, cache, lambd):
    """
    实现我们添加了 L2 正则化的基线模型的反向传播。
    
    参数：
    X -- 输入数据集，形状（输入大小，示例数量）
    Y -- "true" 标签向量，形状（输出大小，示例数量）
    cache -- 缓存 forward_propagation() 的输出
    lambd -- 正则化超参数，标量
    
    返回：
    gradients -- 字典，包含相对于各参数、激活值及预激活值变量的梯度
    """
    
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache
    
    dZ3 = A3 - Y
    #（约1行代码）
    # dW3 = 1./m * np.dot(dZ3, A2.T) + None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)
    
    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    #（约1行代码）
    # dW2 = 1./m * np.dot(dZ2, A1.T) + None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    #（约1行代码）
    # dW1 = 1./m * np.dot(dZ1, X.T) + None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    db1 = 1. / m * np.sum(dZ1, axis=1, keepdims=True)
    
    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1, 
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}
    
    return gradients

In [ ]:
t_X, t_Y, cache = backward_propagation_with_regularization_test_case()

grads = backward_propagation_with_regularization(t_X, t_Y, cache, lambd = 0.7)
print ("dW1 = \n"+ str(grads["dW1"]))
print ("dW2 = \n"+ str(grads["dW2"]))
print ("dW3 = \n"+ str(grads["dW3"]))
backward_propagation_with_regularization_test(backward_propagation_with_regularization)

现在让我们使用 L2 正则化 $(\lambda = 0.7)$ 运行模型。 `model()` 函数将调用： 
- `compute_cost_with_regularization` 而不是 `compute_cost`
- `backward_propagation_with_regularization` 而不是 `backward_propagation`

In [ ]:
parameters = model(train_X, train_Y, lambd = 0.7)
print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

恭喜，测试集准确率提升至 93%。你拯救了法国足球队！

你不再过拟合训练数据。让我们绘制决策边界。

In [ ]:
plt.title("Model with L2-regularization")
axes = plt.gca()
axes.set_xlim([-0.75,0.40])
axes.set_ylim([-0.75,0.65])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

**观察**：
- $\lambda$ 的值是一个超参数，你可以使用开发集进行调整。
- L2 正则化让你的决策边界更加平滑。如果 $\lambda$ 太大，也有可能“过度平滑”，导致模型具有高偏差。

**L2-正则化实际上在做什么？**：

L2-正则化依赖于这样的假设：权重较小的模型比权重较大的模型更简单。因此，通过惩罚代价函数中权重的平方值，你可以将所有权重降低到更小的值。权重大的话代价就变得太高了！这会产生更平滑的模型，其中输出随着输入的变化而变化得更慢。 

<br>
<div style="color:blue;">

**你应该记住的是：** L2-正则化对以下方面的影响：
- 代价计算：
    - 正则化项已添加到代价中。
- 反向传播函数：
    - 梯度中有关于权重矩阵的额外项。
- 权重最终变小（“权重衰减”）： 
    - 权重被推至更小的值。

</div>

<a name='6'></a>
## 6 - Dropout

最后，**Dropout** 是一种广泛使用的正则化技术，专门针对深度学习。 
**它在每次迭代中随机关闭一些神经元。** 观看这两个视频，看看这意味着什么！

<!--
要了解Dropout，请考虑与朋友的对话：
- 朋友：“为什么你需要所有这些神经元来训练你的网络并对图像进行分类？”。 
- 你：“因为每个神经元都包含一个权重，并且可以学习图像的特定特征/细节/形状。我拥有的神经元越多，我的模型学习的特征就越多！”
- 朋友：“我明白了，但是你确定你的神经元正在学习不同的特征，而不是所有相同的特征吗？”
- 你：“好点......同一层中的神经元实际上不会互相交谈。它们绝对有可能学习相同的图像特征/形状/形式/细节......这是多余的。应该有一个解决方案。”
!--> 


<center>
<video width="620" height="440" src="images/dropout1_kiank.mp4" type="video/mp4" controls>
</video>
</center>
<br>
<div style="text-align:center;color:purple;">

<b>图 2 </b>：<b>对第二个隐藏层应用 Dropout。</b> <br>每次迭代时，每个神经元以 $1 - keep\_prob$ 的概率被关闭（置零），或以 $keep\_prob$ 的概率保留（此处为 50%）。被关闭的神经元不参与本次迭代的前向传播和反向传播。

</div>

<center>
<video width="620" height="440" src="images/dropout2_kiank.mp4" type="video/mp4" controls>
</video>
</center>

<div style="text-align:center;color:purple;">

<b> 图 3</b>:<b> 在第一个和第三个隐藏层上Dropout。 </b><br> $1^{st}$ 层：我们平均关闭 40% 的神经元。  $3^{rd}$ 层：我们平均关闭 20% 的神经元。

</div>


当你关闭一些神经元时，你实际上修改了你的模型。 Dropout 背后的想法是，在每次迭代中，你训练一个仅使用神经元子集的不同模型。使用 Dropout，你的神经元对另一个特定神经元的激活值变得不太敏感，因为其他神经元可能随时关闭。 

<a name='6-1'></a>
### 6.1 - 前向传播与 Dropout

<a name='ex-3'></a>
### 练习 3 - forward_propagation_with_dropout

使用 Dropout 实现前向传播。你正在使用 3 层神经网络，并将 Dropout 添加到第一层和第二层隐藏层中。我们不会将 Dropout 应用于输入层或输出层。 

**说明**：
你想关闭第一层和第二层中的一些神经元。为此，你将执行 4 个步骤：
1. 在讲座中，我们讨论了使用 `np.random.rand()` 创建一个与 $a^{[1]}$ 形状相同的变量 $d^{[1]}$ 来随机获取 0 到 1 之间的数字。在这里，你将使用向量化的实现，因此创建一个与以下维度相同的随机矩阵 $D^{[1]} = [d^{[1](1)} d^{[1](2)} ... d^{[1](m)}] $ $A^{[1]}$。
2. 将 $D^{[1]}$ 的每个条目以概率（`keep_prob`）设置为 1，否则设置为 0。

**提示：** 假设 keep_prob = 0.8，这意味着我们想要保留大约 80% 的神经元，并丢弃大约 20% 的神经元。  我们想要生成一个包含 1 和 0 的向量，其中大约 80% 是 1，大约 20% 是 0。
这个Python语句：  
`X = (X < keep_prob).astype(int)`  

在概念上与 if-else 语句相同（对于一维数组的简单情况）：

```
for i,v in enumerate(x):
    if v < keep_prob:
        x[i] = 1
    else: # v >= keep_prob
        x[i] = 0
```
请注意，`X = (X < keep_prob).astype(int)` 适用于多维数组，结果输出保留输入数组的维度。

另请注意，如果不使用 `.astype(int)`，结果是布尔值 `True` 和 `False` 的数组，如果我们将其与数字相乘，Python 会自动将其转换为 1 和 0。  （但是，更好的做法是将数据转换为我们想要的数据类型，因此请尝试使用 `.astype(int)`。）

3. 将 $A^{[1]}$ 设置为 $A^{[1]} * D^{[1]}$。 （你正在关闭一些神经元）。你可以将 $D^{[1]}$ 视为一个掩码，这样当它与另一个矩阵相乘时，它会关闭一些值。
4. 将 $A^{[1]}$ 除以 `keep_prob`，使保留后的激活值在期望上与不使用 Dropout 时一致。这种技术称为反向缩放 Dropout（inverted dropout）。

In [ ]:
# GRADED FUNCTION: forward_propagation_with_dropout

def forward_propagation_with_dropout(X, parameters, keep_prob = 0.5):
    """
    实现前向传播：线性 -> ReLU + Dropout -> 线性 -> ReLU + Dropout -> 线性 -> sigmoid。
    
    参数：
    X -- 输入数据集，形状（2，示例数量）
    parameters -- 包含参数"W1"、"b1"、"W2"、"b2"、"W3"、"b3"的python字典：
                    W1 -- 形状 (20, 2) 的权重矩阵
                    b1 -- 形状为 (20, 1) 的偏置向量
                    W2 -- 形状 (3, 20) 的权重矩阵
                    b2 -- 形状为 (3, 1) 的偏置向量
                    W3 -- 形状 (1, 3) 的权重矩阵
                    b3 -- 形状为 (1, 1) 的偏置向量
    keep_prob - 在Dropout期间保持神经元活跃的概率，标量
    
    返回：
    A3 -- 最后一个激活值，前向传播的输出，形状为 (1,1)
    cache -- 元组，存储用于计算反向传播的信息
    """
    
    np.random.seed(1)
    
    # 检索参数
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]
    
    # 线性 -> ReLU -> 线性 -> ReLU -> 线性 -> sigmoid
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    #（约4行代码） # 下面的步骤1-4对应于上面描述的步骤1-4。 
    # D1 =                                           # 步骤1：初始化矩阵D1 = np.random.rand(..., ...)
    # D1 =                                           # 步骤2：将D1的条目转换为0或1（以keep_prob为阈值）
    # A1 =                                           # 第三步：关闭A1的部分神经元
    # A1 =                                           # 步骤 4：缩放尚未关闭的神经元的值
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    #（约4行代码）
    # D2 =                                           # 步骤1：初始化矩阵D2 = np.random.rand(..., ...)
    # D2 =                                           # 步骤2：将D2的条目转换为0或1（以keep_prob为阈值）
    # A2 =                                           # 第三步：关闭A2的部分神经元
    # A2 =                                           # 步骤 4：缩放尚未关闭的神经元的值
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)
    
    cache = (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)
    
    return A3, cache

In [ ]:
t_X, parameters = forward_propagation_with_dropout_test_case()

A3, cache = forward_propagation_with_dropout(t_X, parameters, keep_prob=0.7)
print ("A3 = " + str(A3))

forward_propagation_with_dropout_test(forward_propagation_with_dropout)

<a name='6-2'></a>
### 6.2 - 反向传播与 Dropout

<a name='ex-4'></a>
### 练习 4 - backward_propagation_with_dropout
使用 Dropout 实现反向传播。和以前一样，你正在训练一个 3 层网络。使用缓存中存储的掩码 $D^{[1]}$ 和 $D^{[2]}$ 将 Dropout 添加到第一和第二个隐藏层。 

**说明**：
反向传播与 Dropout 实际上非常简单。你必须执行 2 个步骤：
1. 你之前在前向传播期间通过将掩码 $D^{[1]}$ 应用于 `A1` 关闭了一些神经元。在反向传播中，你必须通过将相同的掩码 $D^{[1]}$ 重新应用到 `dA1` 来关闭相同的神经元。 
2. 在前向传播期间，你将 `A1` 除以 `keep_prob`。因此，在反向传播中，你必须再次将 `dA1` 除以 `keep_prob`（微积分解释是，如果 $A^{[1]}$ 按 `keep_prob` 缩放，则其导数 $dA^{[1]}$ 也按相同的 `keep_prob` 缩放）。


In [ ]:
# GRADED FUNCTION: backward_propagation_with_dropout

def backward_propagation_with_dropout(X, Y, cache, keep_prob):
    """
    实现我们添加了 Dropout 的基线模型的反向传播。
    
    参数：
    X -- 输入数据集，形状（2，示例数量）
    Y -- "true" 标签向量，形状（输出大小，示例数量）
    cache -- 缓存 forward_propagation_with_dropout() 的输出
    keep_prob - 在Dropout期间保持神经元活跃的概率，标量
    
    返回：
    gradients -- 字典，包含相对于各参数、激活值及预激活值变量的梯度
    """
    
    m = X.shape[1]
    (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3) = cache
    
    dZ3 = A3 - Y
    dW3 = 1./m * np.dot(dZ3, A2.T)
    db3 = 1./m * np.sum(dZ3, axis=1, keepdims=True)
    dA2 = np.dot(W3.T, dZ3)
    #（约2行代码）
    # dA2 =                # 步骤 1：应用掩码 D2 关闭与前向传播期间相同的神经元
    # dA2 =                # 步骤 2：缩放尚未关闭的神经元的值
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1./m * np.dot(dZ2, A1.T)
    db2 = 1./m * np.sum(dZ2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    #（约2行代码）
    # dA1 =                # 步骤 1：应用掩码 D1 关闭与前向传播期间相同的神经元
    # dA1 =                # 步骤 2：缩放尚未关闭的神经元的值
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1./m * np.dot(dZ1, X.T)
    db1 = 1./m * np.sum(dZ1, axis=1, keepdims=True)
    
    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1, 
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}
    
    return gradients

In [ ]:
t_X, t_Y, cache = backward_propagation_with_dropout_test_case()

gradients = backward_propagation_with_dropout(t_X, t_Y, cache, keep_prob=0.8)

print ("dA1 = \n" + str(gradients["dA1"]))
print ("dA2 = \n" + str(gradients["dA2"]))

backward_propagation_with_dropout_test(backward_propagation_with_dropout)

现在让我们使用 Dropout (`keep_prob = 0.86`) 运行模型。这意味着在每次迭代中，你都会以 14% 的概率关闭第 1 层和第 2 层的每个神经元。 函数 `model()` 现在将调用：
- `forward_propagation_with_dropout` 而不是 `forward_propagation`。
- `backward_propagation_with_dropout` 而不是 `backward_propagation`。

In [ ]:
parameters = model(train_X, train_Y, keep_prob = 0.86, learning_rate = 0.3)

print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

Dropout 非常棒！测试准确率再次提升（达到95%）！你的模型没有过拟合训练集，并且在测试集上表现出色。法国足球队将永远感谢你们！ 

运行下面的代码来绘制决策边界。

In [ ]:
plt.title("Model with dropout")
axes = plt.gca()
axes.set_xlim([-0.75,0.40])
axes.set_ylim([-0.75,0.65])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

**注意**：
- 使用 Dropout 时的一个**常见错误**是在训练和测试中都使用它。你应该仅在训练中使用 Dropout（随机消除节点）。 
- [TensorFlow](https://www.tensorflow.org/api_docs/python/tf/nn/dropout)、[桨桨](https://www.paddlepaddle.org.cn/documentation/docs/en/api/paddle/nn/Dropout_en.html#dropout)、[Keras](https://keras.io/api/layers/regularization_layers/dropout/) 或 [咖啡厅](https://caffe.berkeleyvision.org/doxygen/classcaffe_1_1DropoutLayer.html) 等深度学习框架附带 Dropout 层实现。不要紧张 - 你很快就会学习其中一些框架。

<div style="color:blue;">

**关于 Dropout 你应该记住什么：**
- Dropout 是一种正则化技术。
- 你仅在训练期间使用Dropout。测试期间不要使用Dropout（随机消除节点）。
- 在前向传播与反向传播期间应用 Dropout。
- 在训练期间，将每个 Dropout 层除以 keep_prob，以保持激活值的期望值相同。例如，如果 keep_prob 为 0.5，那么我们平均将关闭一半节点，因此输出将缩放 0.5，因为只有剩下的一半对解决方案有贡献。除以 0.5 相当于乘以 2。因此，输出现在具有相同的期望值。即使 keep_prob 是 0.5 以外的其他值，你也可以检查这是否有效。

</div>

<a name='7'></a>
## 7 - 结论

**以下是我们三个模型的结果**： 

<table> 
    <tr>
        <td>
        <b>模型</b>
        </td>
        <td>
        <b>训练精度</b>
        </td>
        <td>
        <b>测试精度</b>
        </td>
    </tr>
        <td>
        不带正则化的 3 层神经网络
        </td>
        <td>
        95%
        </td>
        <td>
        91.5%
        </td>
    <tr>
        <td>
        具有 L2-正则化的 3 层神经网络
        </td>
        <td>
        94%
        </td>
        <td>
        93%
        </td>
    </tr>
    <tr>
        <td>
        采用 Dropout 的 3 层神经网络
        </td>
        <td>
        93%
        </td>
        <td>
        95%
        </td>
    </tr>
</table> 

请注意，正则化会损害训练集的性能！这是因为它限制了网络过拟合训练集的能力。但由于它最终提供了更好的测试精度，因此对你的系统很有帮助。 

恭喜你完成了这个作业！也为法国足球带来了革命性的变化。 :-) 

<div style="color:blue;">

**我们希望你从 Notebook 中记住什么**：
- 正则化将帮助你减少过拟合。
- 正则化会将你的体重降低到更低的值。
- L2 正则化和 Dropout 是两种非常有效的正则化技术。

</div>